## Student Attendance Analytics with Pandas

In [ ]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [ ]:
df = pd.read_excel("csec_attendance.xlsx")
df

#### Section 1: Data Inspection

In [ ]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""

display(df.head(10))
print("Shape:",df.shape)
print("DataFrame:",df.dtypes)


In [ ]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """


null_counts = df.isna().sum()
print("Null values by column:")
print(null_counts)


print("\nColumns with null values:")
print(null_counts[null_counts > 0])


never_count = df["Last Present Date"].eq("Never").sum()
print(f'\n"Never" values in Last Present Date: {never_count}')

In [ ]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""


df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace("%", "pct", regex=False)
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
print(df.columns.tolist())

In [ ]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""

df["attendance_pct"] = pd.to_numeric(
    df["attendance_pct"].astype("string").str.rstrip("%"),
    errors="coerce"
)

df["last_present_date"] = pd.to_datetime(
    df["last_present_date"].replace("Never", pd.NA),
    errors="coerce"
)

display(df[["attendance_pct", "last_present_date"]].head(10))

#### Section 2: Filtering & Selection

In [ ]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""

critical_students = (
    df[df["status"].eq("Critical")]
    .sort_values("attendance_pct", ascending=True)
)

display(critical_students)

In [ ]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""

print(df.columns.tolist())
fully_absent = df.loc[
    df["present"].eq(0),
    ["roll_no", "student_name", "consecutive_absences"]
]

display(fully_absent)

In [ ]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """


safe_students = df.loc[df["status"].eq("Safe")]
display(safe_students)

display(df.iloc[:15, 1:6])

In [ ]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""

students_with_streak = df.loc[
    df["consecutive_absences"].ge(5),
    ["student_name", "status", "last_present_date"]
]

display(students_with_streak)

### Section 3: Grouping & Aggregation 

In [ ]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""

status_summary = (
    df.groupby("status")
    .agg(
        student_count=("student_name", "count"),
        average_attendance_pct=("attendance_pct", "mean"),
        max_consecutive_absences=("consecutive_absences", "max")
    )
    .reset_index()
)

display(status_summary)

In [ ]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""

df["attendance_category"] = pd.cut(
    df["attendance_pct"],
    bins=[float("-inf"), 60, 75, 90, float("inf")],
    labels=["Poor", "Average", "Good", "Excellent"],
    right=False
)

category_counts = (
    df.groupby("attendance_category", observed=True)
    .size()
    .reset_index(name="student_count")
)

display(category_counts)


In [ ]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""

totals = df.agg(
    total_classes_conducted=("total_classes", "sum"),
    total_present=("present", "sum"),
    total_absent=("absent", "sum")
)

display(totals)


In [ ]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""

status_stats = (
    df.groupby("status")
    .agg(
        present_min=("present", "min"),
        present_max=("present", "max"),
        present_mean=("present", "mean"),
        absent_sum=("absent", "sum")
    )
)

display(status_stats)                                

### Section 4: Feature Engineering & Transformation

In [ ]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""

df["risk_score"] = (
    df["absent"] * 0.4
    + df["consecutive_absences"] * 0.6
)

top_10_risk = df.nlargest(
    10, "risk_score"
)[["roll_no", "student_name", "risk_score"]]

display(top_10_risk)


In [ ]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""

df["classes_needed_numeric"] = (
    pd.to_numeric(
        df["classes_needed_75pct"].astype("string").str.extract(r"(\d+)")[0],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

display(df[["classes_needed_75pct", "classes_needed_numeric"]].head(10))

In [ ]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """

df["attendance_flag"] = df["attendance_pct"].apply(
    lambda attendance: "Defaulter" if attendance < 75 else "Safe"
)

display(df[["attendance_pct", "attendance_flag"]].head(10))

In [ ]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""

df["attendance_range"] = pd.cut(
    df["attendance_pct"],
    bins=[0, 50, 75, 90, 100],
    labels=["Critical", "Warning", "Good", "Excellent"],
    include_lowest=True
)

display(df[["attendance_pct", "attendance_range"]].head(10))


### Section 5: Advanced Analysis 

In [ ]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""

summary = pd.pivot_table(
    df,
    index="status",
    columns="attendance_category",
    values="student_name",
    aggfunc="count",
    fill_value=0,
    observed=True
)

display(summary)

In [ ]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""

base_name = (
    df["student_name"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s*[\(\-_]?\s*\d+\)?$", "", regex=True)
    .str.strip()
)

duplicate_mask = base_name.duplicated(keep=False)

display(
    df.loc[duplicate_mask]
    .assign(base_name=base_name[duplicate_mask])
    .sort_values("base_name")
)


In [ ]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""

Student_Details = df[["roll_no"]].drop_duplicates().copy()
Student_Details["Branch"] = "CSE"   # placeholder
Student_Details["Semester"] = 1     # placeholder

student_details = Student_Details.rename(columns={
    "Branch": "branch",
    "Semester": "semester"
})

merged_df = df.merge(student_details, on="roll_no", how="inner")
display(merged_df)

In [ ]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""


critical_students = df.loc[df["status"].eq("Critical")]
print(critical_students)

status_summary = (
    df.groupby("status")
    .agg(
        student_count=("student_name", "count"),
        average_attendance_pct=("attendance_pct", "mean"),
        max_consecutive_absences=("consecutive_absences", "max")
    )
    .reset_index()
)
print(status_summary)


top_10_risk = df.nlargest(10, "risk_score")
print(top_10_risk)


critical_students.to_csv("critical_students.csv", index=False)
status_summary.to_csv("status_summary.csv", index=False)
top_10_risk.to_csv("top_10_risk.csv", index=False)

print("CSV files saved.")